In [13]:
!hostname

cn002.delta.ncsa.illinois.edu


In [14]:
import moscot as mc

In [15]:
import scanpy as sc
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
import os
os.chdir('/projects/bgdb/asachan/methods/maxToki-multimodal')  # directory containing utils.py
import sys
import logging
import warnings

export_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
raw_dir = "/work/hdd/bgdb/asachan/datasets_hdd/SKM_multimodal_ageing/raw_files/female_type2_maxtoki"

In [17]:
adata_rna = '/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human/rna_objects/rna_female_type2.h5ad'
out_tmp = '/projects/bgdb/asachan/tmp'

## Downsampling (not random, based on dna-damage and metabolic scoring trends, so that we can focus on signals of interest)

In [18]:
rna_adata = sc.read_h5ad(adata_rna)
# filter out OM2 cells
rna_adata = rna_adata[rna_adata.obs['sample'] != 'OM2'].copy()
# filter out YM2_1 cells
rna_adata = rna_adata[rna_adata.obs['orig.ident'] != 'YM2_1'].copy()

In [19]:
#drop OM9_D* cells from rna_adata (these are Gluteus Medius cells)
rna_adata = rna_adata[~rna_adata.obs['orig.ident'].str.contains('OM9_D')]

In [20]:
rna_adata.obs['orig.ident'].value_counts()

orig.ident
OM9_N1    2324
OM9_N2    2290
OM9_N4    1526
OM9_N3    1499
YM2_2     1139
OM6_3      978
OM6_4      937
YM2_3      850
OM6_2      830
OM6_1      691
P26_2       45
P26_5       40
P26_4       26
P26_1       11
Name: count, dtype: int64

In [21]:
rna_adata.obs['sample'].value_counts()

sample
OM9    7639
OM6    3436
YM2    1989
P26     122
Name: count, dtype: int64

# adding raw counts layer for these retained cells

In [22]:
rna_adata

View of AnnData object with n_obs × n_vars = 13186 × 48355
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'percent.mt', 'age', 'tech', 'Sex', 'Country', 'age_pop', 'Annotation', 'Pseudotime', 'Pseudotime_typeI', 'Pseudotime_typeII'
    var: 'features'
    uns: 'Annotation_colors', 'Final_annotation2_colors', 'Final_annotation3_colors', 'Final_annotation4_colors', 'Final_annotation_colors', 'age_pop_colors', 'anno_0713_colors', 'anno_0715_colors', 'fiber_class_V1_colors', 'integrated_snn_res.0.8_colors', 'integrated_snn_res.2.5_colors', 'integrated_snn_res.2_colors', 'integrated_snn_res.3_colors', 'integrated_snn_res.7_colors', 'orig.ident_colors', 'rank_genes_groups'
    obsm: 'X_umap'

In [23]:
rna_adata.X.max() # float means the counts are normalized

np.float64(4.331152716932476)

#### Suffix per replicate library

In [28]:
import re

pat = re.compile(r'^(CELL\d+_N\d+)(_.+)$')

# 1. Find merge suffix per orig.ident
suffix_per_orig = {}
for oid in rna_adata.obs['orig.ident'].unique():
    cells = rna_adata.obs_names[rna_adata.obs['orig.ident'] == oid]
    suffixes = {pat.match(c).group(2) for c in cells}
    assert len(suffixes) == 1, f"{oid}: multiple suffixes {suffixes}"
    suffix_per_orig[oid] = suffixes.pop()

# 2. Strip the suffix → set of raw-style barcodes per orig.ident
oid_to_stripped = {}
for oid, suf in suffix_per_orig.items():
    cells = rna_adata.obs_names[rna_adata.obs['orig.ident'] == oid]
    oid_to_stripped[oid] = {c[:-len(suf)] for c in cells}

In [29]:
def load_raw_barcodes(path):
    return pd.read_csv(path, header=None, sep='\t')[0].tolist()

In [ ]:
# Quick check of barcodes in adata being a complete subset of barcodes in raw library files (P26, YM2, OM6, OM9)
sample_name = 'OM9'
for oid in [f'{sample_name}_N1', f'{sample_name}_N2', f'{sample_name}_N3', f'{sample_name}_N4']:
    print(oid, len(oid_to_stripped[oid]),
          'cells:', list(oid_to_stripped[oid])[:3])

sample_raw_dir = f"{raw_dir}/{sample_name}"
sample_libs = [f'{sample_name}_VL_snRNA_seq_1', f'{sample_name}_VL_snRNA_seq_2',
            f'{sample_name}_VL_snRNA_seq_3', f'{sample_name}_VL_snRNA_seq_4']
sample_oids = [f'{sample_name}_N1', f'{sample_name}_N2', f'{sample_name}_N3', f'{sample_name}_N4']

# Load raw barcode sets
raw_sample = {}
for lib in sample_libs:
    raw_sample[lib] = set(load_raw_barcodes(f"{sample_raw_dir}/{lib}_barcodes.tsv.gz"))
    print(f"{lib}: {len(raw_sample[lib])} barcodes")

# 4 x 4 subset table: rows = orig.ident, cols = raw library
rows = []
for oid in sample_oids:
    expected = oid_to_stripped[oid]   # adata cells (suffix-stripped) for this orig.ident
    row = {'orig.ident': oid, 'n_adata': len(expected)}
    for lib in sample_libs:
        n = len(expected & raw_sample[lib])
        row[lib] = f"{n}/{len(expected)}"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

OM9_N1 2324 cells: ['CELL1975_N1', 'CELL4426_N1', 'CELL5235_N1']
OM9_N2 2290 cells: ['CELL4989_N1', 'CELL8202_N1', 'CELL7105_N1']
OM9_N3 1499 cells: ['CELL76_N2', 'CELL812_N3', 'CELL3783_N1']
OM9_N4 1526 cells: ['CELL4579_N1', 'CELL7105_N1', 'CELL3182_N1']
OM9_VL_snRNA_seq_1: 10924 barcodes
OM9_VL_snRNA_seq_2: 10531 barcodes
OM9_VL_snRNA_seq_3: 9131 barcodes
OM9_VL_snRNA_seq_4: 7788 barcodes
orig.ident  n_adata OM9_VL_snRNA_seq_1 OM9_VL_snRNA_seq_2 OM9_VL_snRNA_seq_3 OM9_VL_snRNA_seq_4
    OM9_N1     2324          2324/2324          2143/2324          1774/2324          1693/2324
    OM9_N2     2290          2231/2290          2290/2290          1707/2290          1640/2290
    OM9_N3     1499          1174/1499          1076/1499          1499/1499          1275/1499
    OM9_N4     1526          1297/1526          1196/1526          1308/1526          1526/1526


In [33]:
# orig.ident → raw library file prefix
# Fill this in once based on your knowledge of the file naming
orig_ident_to_lib = {
    'OM6_1':  'OM6_VL_snRNA_seq_1',
    'OM6_2':  'OM6_VL_snRNA_seq_2',
    'OM6_3':  'OM6_VL_snRNA_seq_3',
    'OM6_4':  'OM6_VL_snRNA_seq_4',
    'OM9_N1': 'OM9_VL_snRNA_seq_1',   # adjust to whatever the actual filenames are
    'OM9_N2': 'OM9_VL_snRNA_seq_2',
    'OM9_N3': 'OM9_VL_snRNA_seq_3',
    'OM9_N4': 'OM9_VL_snRNA_seq_4',
    'YM2_2':  'YM2_ST_snRNA_seq_2',
    'YM2_3':  'YM2_ST_snRNA_seq_3',
    'P26_1': 'P26_ST_snRNA_seq_1',
    'P26_2': 'P26_ST_snRNA_seq_2',
    'P26_4': 'P26_ST_snRNA_seq_3',
    'P26_5': 'P26_ST_snRNA_seq_4',
}
assert set(orig_ident_to_lib) == set(suffix_per_orig), \
    f"Mapping mismatch: {set(suffix_per_orig) ^ set(orig_ident_to_lib)}"